# 01 Build Modality Benchmark

Objective: convert the reviewed seed capabilities into controlled modality minimal pairs and gold labels for the two tasks.


In [ ]:
from pathlib import Path
import os
import sys
import importlib

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
while not (PROJECT_ROOT / "AGENTS.md").exists() and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.insert(0, str(PROJECT_ROOT / "scripts"))
import eval_utils as eu
eu = importlib.reload(eu)

CONFIG_PATH = PROJECT_ROOT / "config.json"
if not CONFIG_PATH.exists():
    CONFIG_PATH = PROJECT_ROOT / "config.example.json"
CONFIG = eu.load_config(CONFIG_PATH)
eu.ensure_project_dirs(PROJECT_ROOT)
BENCHMARK_VARIANT = os.getenv("BENCHMARK_VARIANT", "must").strip().lower()
VARIANT_SUFFIX = eu.variant_suffix(BENCHMARK_VARIANT)

PROJECT_ROOT, CONFIG_PATH, BENCHMARK_VARIANT


## Load Reviewed Seeds


In [ ]:
target_count = int(CONFIG["project"]["target_seed_count"])
review_path = PROJECT_ROOT / "data/processed/seeds_review.csv"
seeds = eu.load_reviewed_seeds(review_path, target_count=target_count, strict=True)
print(f"Loaded {len(seeds)} reviewed seeds.")
seeds[:2]


## Generate Four Modality Variants Per Seed


In [ ]:
benchmark = eu.build_benchmark_items(seeds)
benchmark_path = PROJECT_ROOT / "data/processed/benchmark_items.csv"
candidate_benchmark_path = PROJECT_ROOT / "data/processed/benchmark_items_candidate.csv"

# Existing benchmark artifacts are preserved by default after review.
# Set FORCE_REBUILD_BENCHMARK = True only when you intentionally accept regenerated items.
FORCE_REBUILD_BENCHMARK = False
write_result = eu.write_csv_rows_if_changed(
    benchmark_path,
    benchmark,
    candidate_path=candidate_benchmark_path,
    overwrite=FORCE_REBUILD_BENCHMARK,
)

if write_result["status"] == "written":
    print(f"Wrote benchmark: {benchmark_path}")
elif write_result["status"] == "overwritten":
    print(f"Overwrote benchmark by explicit request: {benchmark_path}")
elif write_result["status"] == "unchanged":
    print(f"Existing benchmark matches regenerated items: {benchmark_path}")
else:
    print(f"Existing benchmark preserved: {benchmark_path}")
    print(f"Regenerated candidate differs and was written for review: {write_result['candidate_path']}")

benchmark = eu.read_csv_rows(benchmark_path)
print(f"Items: {len(benchmark)}")
benchmark[:4]


## Label and Shape Checks


In [ ]:
expected_items = target_count * len(eu.MODALITIES)
assert len(benchmark) == expected_items, (len(benchmark), expected_items)
assert len({row["item_id"] for row in benchmark}) == expected_items
assert all(row["task1_gold_decision"] == ("yes" if row["source_modality"] == "mandatory" else "no") for row in benchmark)
assert all(row["task2_gold_modality"] == row["source_modality"] for row in benchmark)
assert eu.ORDINAL_STRENGTH["mandatory"] > eu.ORDINAL_STRENGTH["recommended"] > eu.ORDINAL_STRENGTH["optional"] > eu.ORDINAL_STRENGTH["nice_to_have"]
print("OK: benchmark shape, labels, and modality ordering are valid.")


## Export Benchmark Statements For Review


In [ ]:
import pandas as pd

benchmark_review = eu.benchmark_statement_review_frame(benchmark)
review_paths = eu.write_benchmark_statement_review(benchmark, PROJECT_ROOT / "outputs")

print(f"Review rows: {len(benchmark_review)}")
print(f"Wrote Markdown review table: {review_paths['markdown']}")
print(f"Wrote CSV review table: {review_paths['csv']}")

with pd.option_context("display.max_rows", 20, "display.max_colwidth", 160, "display.width", 240):
    display(benchmark_review.head(20))


## Build SHALL Robustness Benchmark


In [ ]:
shall_benchmark = eu.build_benchmark_items(seeds, mandatory_keyword="SHALL")
shall_path = PROJECT_ROOT / "data/processed/benchmark_items_shall.csv"
candidate_shall_path = PROJECT_ROOT / "data/processed/benchmark_items_shall_candidate.csv"

FORCE_REBUILD_SHALL_BENCHMARK = False
shall_write_result = eu.write_csv_rows_if_changed(
    shall_path,
    shall_benchmark,
    candidate_path=candidate_shall_path,
    overwrite=FORCE_REBUILD_SHALL_BENCHMARK,
)

if shall_write_result["status"] == "written":
    print(f"Wrote SHALL benchmark: {shall_path}")
elif shall_write_result["status"] == "overwritten":
    print(f"Overwrote SHALL benchmark by explicit request: {shall_path}")
elif shall_write_result["status"] == "unchanged":
    print(f"Existing SHALL benchmark matches regenerated items: {shall_path}")
else:
    print(f"Existing SHALL benchmark preserved: {shall_path}")
    print(f"Regenerated candidate differs and was written for review: {shall_write_result['candidate_path']}")

shall_benchmark = eu.read_csv_rows(shall_path)
assert len(shall_benchmark) == expected_items
assert len({row["item_id"] for row in shall_benchmark}) == expected_items
assert all(row["task1_gold_decision"] == ("yes" if row["source_modality"] == "mandatory" else "no") for row in shall_benchmark)
assert all(row["task2_gold_modality"] == row["source_modality"] for row in shall_benchmark)
assert all("SHALL" in row["source_statement"] for row in shall_benchmark if row["source_modality"] == "mandatory")
assert all("SHALL" in row["candidate_requirement"] for row in shall_benchmark)

shall_review_paths = eu.write_benchmark_statement_review(shall_benchmark, PROJECT_ROOT / "outputs", suffix="_shall")
print(f"SHALL items: {len(shall_benchmark)}")
print(f"Wrote SHALL Markdown review table: {shall_review_paths['markdown']}")
print(f"Wrote SHALL CSV review table: {shall_review_paths['csv']}")


## Write Benchmark Manifest


In [ ]:
manifest_path = PROJECT_ROOT / "outputs/benchmark_manifest.json"
manifest_paths = [
    PROJECT_ROOT / "data/processed/seeds_review.csv",
    PROJECT_ROOT / "data/processed/seeds_selected.csv",
    PROJECT_ROOT / "data/processed/benchmark_items.csv",
    PROJECT_ROOT / "data/processed/benchmark_items_shall.csv",
    PROJECT_ROOT / "prompts/mandatory_entailment.txt",
    PROJECT_ROOT / "prompts/mandatory_entailment_strict.txt",
    PROJECT_ROOT / "prompts/modality_extraction.txt",
    PROJECT_ROOT / "prompts/modality_extraction_labels_only.txt",
]
manifest = eu.write_benchmark_manifest(
    manifest_paths,
    manifest_path,
    root=PROJECT_ROOT,
    metadata={
        "main_benchmark": "MUST",
        "robustness_benchmark": "SHALL",
        "seed_count": target_count,
        "source_modalities": eu.MODALITIES,
    },
)
print(f"Wrote manifest: {manifest_path}")
print(f"Artifacts recorded: {len(manifest['artifacts'])}")
